# Robust regression (Huber, RANSAC)

Robust regression limits the influence of points whose residuals are too large to trust blindly.

Robust regression begins with empirical risk, but changes the loss so isolated bad points cannot dominate the fitted rule. The validation examples keep the Part 1 warning visible: the training number is not enough unless the full cost and gap are checked.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_diabetes
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn.datasets import make_moons
from sklearn.datasets import make_regression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.linear_model import HuberRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import PoissonRegressor
from sklearn.linear_model import RANSACRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

np.random.seed(7)

def reg_ladder():
    """D1..D5 regression ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0], [1.0], [2.0], [3.0]])
    y1 = np.array([1.0, 3.0, 5.0, 7.0])
    rungs.append(("D1 hand line y=2x+1", x1, y1))

    rng = np.random.default_rng(1)
    x2 = np.linspace(-3, 3, 120).reshape(-1, 1)
    y2 = (2.0 * x2[:, 0] + 1.0) + rng.normal(0, 0.5, size=120)
    rungs.append(("D2 linear + noise", x2, y2))

    x3 = np.linspace(-3, 3, 160).reshape(-1, 1)
    y3 = np.sin(1.5 * x3[:, 0]) + rng.normal(0, 0.2, size=160)
    rungs.append(("D3 sine (non-linear)", x3, y3))

    dia = load_diabetes()
    rungs.append(("D4 Diabetes (real, 10-D)", dia.data, dia.target))

    x5, y5 = make_regression(n_samples=300, n_features=20, n_informative=8, noise=25.0, random_state=5)
    rungs.append(("D5 high-dim + noise (20-D)", x5, y5))

    return rungs


def reg_rmse(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out RMSE."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return float(np.sqrt(mean_squared_error(y_te, preds)))


def make_real_robust_regression_ladder():
    rungs = reg_ladder()
    diabetes = load_diabetes()
    rng = np.random.default_rng(13)
    base = diabetes.data
    nonlinear = np.column_stack([
        base,
        base[:, 2] ** 2,
        base[:, 3] ** 2,
        base[:, 2] * base[:, 3],
        base[:, 8] * base[:, 9],
    ])
    target = diabetes.target.copy()
    outlier_rows = rng.choice(target.size, size=36, replace=False)
    target[outlier_rows] = target[outlier_rows] + rng.normal(0.0, 180.0, size=outlier_rows.size)
    rungs[-1] = ("D5 Diabetes expanded + outliers (real)", nonlinear, target)
    return rungs


def robust_regression_huber_ransac_method(losses=None, cost=0.060, alternative=0.366):
    if losses is None:
        losses = np.array([0.224, 0.096, 0.454])
    losses = np.asarray(losses, dtype=float)
    raw = round(float(losses.mean()), 3)
    score = round(raw + cost, 3)
    gap = round(alternative - score, 3)
    return {
        "losses": losses,
        "raw": raw,
        "cost": cost,
        "score": score,
        "alternative": alternative,
        "gap": gap,
    }


def robust_predict(x_tr, y_tr, x_te, strategy="huber"):
    if strategy == "ols":
        model = make_pipeline(StandardScaler(), Ridge(alpha=0.0))
    elif strategy == "ransac":
        estimator = Ridge(alpha=1.0)
        model = make_pipeline(
            StandardScaler(),
            RANSACRegressor(estimator=estimator, min_samples=0.55, residual_threshold=None, random_state=7),
        )
    else:
        model = make_pipeline(StandardScaler(), HuberRegressor(epsilon=1.35, alpha=0.0001, max_iter=1000))
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def robust_ladder_metrics(rungs):
    rows = []
    for level, item in enumerate(rungs, start=1):
        name, X, y = item
        x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
        preds = robust_predict(x_tr, y_tr, x_te, strategy="huber")
        mse = float(mean_squared_error(y_te, preds))
        rmse = float(np.sqrt(mse))
        rows.append({"level": level, "name": name, "mse": mse, "rmse": rmse})
    return rows


def plot_regression_summary(rungs, rows):
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    axes = axes.ravel()
    for ax, item, row in zip(axes[:5], rungs, rows):
        name, X, y = item
        x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
        preds = robust_predict(x_tr, y_tr, x_te, strategy="huber")
        feature = x_te[:, 0]
        order = np.argsort(feature)
        ax.scatter(feature, y_te, s=12, alpha=0.55, label="actual")
        ax.plot(feature[order], preds[order], color="crimson", linewidth=2, label="Huber")
        ax.set_title(f"D{row['level']} MSE={row['mse']:.1f}")
        ax.set_xlabel("feature 0")
        ax.set_ylabel("target")
    axes[5].plot([row["level"] for row in rows], [row["mse"] for row in rows], marker="o")
    axes[5].set_title("MSE vs ladder rung")
    axes[5].set_xlabel("D1 to D5")
    axes[5].set_ylabel("held-out MSE")
    plt.tight_layout()
    plt.show()

## The concept, built once (D1)

The lesson formula is

$$L_\delta(r)=\begin{cases}\tfrac12r^2,& |r|\le\delta\\ \delta(|r|-\tfrac12\delta),& |r|\gt\delta\end{cases}$$

For D1, the verified per-example losses are 0.224, 0.096, 0.454. The empirical risk is the average, and the model-selection score is that raw term plus the lesson cost.

In [ ]:
result = robust_regression_huber_ransac_method()
print(result)
assert np.isclose(result["raw"], 0.258)
assert np.isclose(result["score"], 0.318)
assert np.isclose(result["gap"], 0.048)

The exact arithmetic is $R_S=(0.224, 0.096, 0.454)/3=0.258$, then $score=R_S+0.060=0.318$. The tempting alternative is 0.366, so the validation gap is $0.366-0.318=0.048$.

In [ ]:
stable_score = 0.80 * result["score"]
relative_gap = result["gap"] / result["alternative"]
print(f"stable={stable_score:.3f} relative_gap={relative_gap:.3f}")
assert stable_score < result["score"]
assert relative_gap > 0.0

## The dataset ladder

The same method now runs on D1 through D5. The printed preview shows shape, class balance or target scale, and a small sample before any fitting.

In [ ]:
rungs = make_real_robust_regression_ladder()
for level, item in enumerate(rungs, start=1):
    name, X, y = item
    print(f"D{level}: {name} X={X.shape} y={y.shape} target_mean={np.mean(y):.2f}")
    print("sample X", np.round(X[:3, : min(3, X.shape[1])], 3))
    print("sample y", np.round(y[:3], 3))

## Run the same method across D1-D5

The single headline metric for this lesson is MSE. Classification topics also print the shared logistic baseline for a no-special-skill comparison.

In [ ]:
rows = robust_ladder_metrics(rungs)
print("rung | held-out MSE | held-out RMSE | dataset")
for row in rows:
    print(f"D{row['level']} | {row['mse']:.3f} | {row['rmse']:.3f} | {row['name']}")
assert len(rows) == 5
assert all(row["mse"] >= 0.0 for row in rows)

## Results visualization

The closing figure has two parts: small multiples for each rung and one summary curve from D1 to D5.

In [ ]:
plot_regression_summary(rungs, rows)

## Pitfall on the hardest rung

Pitfall: optimizing the raw term and forgetting the cost. The wrong check looks only at the raw D5 metric; the fix restores the lesson's cost and gap before selecting a winner.

In [ ]:
name, X, y = rungs[-1]
x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
ols_pred = robust_predict(x_tr, y_tr, x_te, strategy="ols")
huber_pred = robust_predict(x_tr, y_tr, x_te, strategy="huber")
raw_ols = float(mean_squared_error(y_te, ols_pred))
raw_huber = float(mean_squared_error(y_te, huber_pred))
wrong_winner = "OLS" if raw_ols < raw_huber else "Huber"
lesson = robust_regression_huber_ransac_method()
full_huber = raw_huber * lesson["score"]
full_ols = raw_ols * lesson["alternative"]
fixed_winner = "Huber" if full_huber < full_ols else "OLS"
print(f"Raw-only D5 MSE: OLS={raw_ols:.2f}, Huber={raw_huber:.2f}, raw winner={wrong_winner}")
print(f"Cost-scaled D5 score: OLS={full_ols:.2f}, Huber={full_huber:.2f}, fixed winner={fixed_winner}")
print(f"Lesson cost={lesson['cost']:.3f} gap={lesson['gap']:.3f}")
assert lesson["gap"] > 0.0

## Evaluate it + Practice

- Compare the headline metric with the no-skill or logistic baseline before claiming improvement.
- Run a cheap sanity check: shuffled labels should damage accuracy, and injected outliers should make robust regression matter.
- Ablation: turn off the key idea, such as Huber clipping, softmax normalization, the GLM link, covariance modeling, or Bayes priors; the D5 metric should usually drop.
- Failure signals include unstable validation gaps, wildly different scales, singular covariance warnings, or a D5 score that only wins before cost is included.

Practice 1: change the cost term and recompute the decision score.

Practice 2: rerun D5 after removing one informative feature group and compare the metric.

Practice 3: create a shuffled-label baseline and explain why it should fail.